# Data Center NPV Analysis

This notebook analyzes three scenarios:
- **Scenario A**: Metro Baseline (No Flexibility, No Battery)
- **Scenario B**: Rural with Flexible Load
- **Scenario C**: Rural with Battery Energy Storage System (BESS)

In [9]:
import numpy_financial as npf

def enhanced_scenario_analysis(name, discount_rate, initial_capex, period_years,
                                year_1_revenue, year_1_opex,
                                revenue_inflation_rate, opex_inflation_rate,
                                target_irr=None):
    
    def run_npv_irr_calc(year_1_rev):
        cash_flows = []
        annual_opex_list = []
        for year in range(1, period_years + 1):
            rev = year_1_rev * ((1 + revenue_inflation_rate) ** (year - 1))
            opex = year_1_opex * ((1 + opex_inflation_rate) ** (year - 1))
            cash_flows.append(rev - opex)
            annual_opex_list.append(opex)

        all_cash_flows = [-initial_capex] + cash_flows
        npv = npf.npv(discount_rate, all_cash_flows)
        irr = npf.irr(all_cash_flows)
        tco = initial_capex + sum(annual_opex_list)
        total_noi = sum(cash_flows)

        # Discounted Payback
        payback = -1
        cumulative = -initial_capex
        for i, cf in enumerate(cash_flows):
            cumulative += cf / ((1 + discount_rate) ** (i + 1))
            if cumulative > 0:
                payback = i + 1
                break

        return {
            "cash_flows": cash_flows,
            "npv": npv,
            "irr": irr,
            "tco": tco,
            "total_noi": total_noi,
            "payback": payback
        }

    # Step 1: Evaluate original revenue assumption
    original_result = run_npv_irr_calc(year_1_revenue)

    # Step 2: Calculate required revenue for target IRR
    if target_irr:
        from scipy.optimize import minimize_scalar

        def irr_gap(rev):
            result = run_npv_irr_calc(rev)
            return abs(result["irr"] - target_irr)

        res = minimize_scalar(irr_gap, bounds=(1e6, 1e8), method='bounded')
        required_revenue = res.x
        target_result = run_npv_irr_calc(required_revenue)
    else:
        required_revenue = year_1_revenue
        target_result = original_result

    # Step 3: Print full comparison
    print(f"\n=== Financial Analysis for: {name} ===")
    print(f"--- Based on Original Revenue Assumption (${year_1_revenue:,.2f}) ---")
    print(f"Net Present Value (NPV):         ${original_result['npv']:,.2f}")
    print(f"Internal Rate of Return (IRR):    {original_result['irr']:.2%}")
    print(f"Total Cost of Ownership (TCO):   ${original_result['tco']:,.2f}")
    print(f"Total Net Operating Income:      ${original_result['total_noi']:,.2f}")
    print("Discounted Payback Period:       ",
          f"{original_result['payback']} years" if original_result['payback'] != -1 else "Does not pay back")

    if target_irr:
        print(f"\n--- Revenue Needed to Achieve {target_irr:.0%} IRR ---")
        print(f"Required Revenue:                ${required_revenue:,.2f}")
        print(f"Net Present Value (NPV):         ${target_result['npv']:,.2f}")
        print(f"Internal Rate of Return (IRR):    {target_result['irr']:.2%}")
        print(f"Revenue Gap:                     ${required_revenue - year_1_revenue:,.2f}")
        print(f"Discounted Payback Period:       ",
              f"{target_result['payback']} years" if target_result['payback'] != -1 else "Does not pay back")
    print("=" * 60)

    return original_result, target_result, required_revenue


In [10]:
scenario_a_current, scenario_a_target, rev_a = enhanced_scenario_analysis(
    name="Scenario A: Metro Baseline",
    discount_rate=0.08,
    initial_capex=277950000,
    period_years=10,
    year_1_revenue=35000000,
    year_1_opex=28968540,
    revenue_inflation_rate=0.0,
    opex_inflation_rate=0.03,
    target_irr=0.10
)



=== Financial Analysis for: Scenario A: Metro Baseline ===
--- Based on Original Revenue Assumption ($35,000,000.00) ---
Net Present Value (NPV):         $-261,813,500.87
Internal Rate of Return (IRR):    nan%
Total Cost of Ownership (TCO):   $610,041,846.39
Total Net Operating Income:      $17,908,153.61
Discounted Payback Period:        Does not pay back

--- Revenue Needed to Achieve 10% IRR ---
Required Revenue:                $77,688,468.41
Net Present Value (NPV):         $24,629,596.93
Internal Rate of Return (IRR):    10.00%
Revenue Gap:                     $42,688,468.41
Discounted Payback Period:        9 years


In [11]:
# Run Scenario B financial analysis using target IRR logic
scenario_b_current, scenario_b_target, rev_b  = enhanced_scenario_analysis(
    name="Scenario B: Rural Flexible Load",
    discount_rate=0.08,
    initial_capex=183_450_000,
    period_years=10,
    year_1_revenue=30_000_000,     # Original assumption
    year_1_opex=25_912_165,
    revenue_inflation_rate=0.0,
    opex_inflation_rate=0.03,
    target_irr=0.10                 # Target IRR to test
)



=== Financial Analysis for: Scenario B: Rural Flexible Load ===
--- Based on Original Revenue Assumption ($30,000,000.00) ---
Net Present Value (NPV):         $-177,787,868.01
Internal Rate of Return (IRR):    nan%
Total Cost of Ownership (TCO):   $480,503,932.26
Total Net Operating Income:      $2,946,067.74
Discounted Payback Period:        Does not pay back

--- Revenue Needed to Achieve 10% IRR ---
Required Revenue:                $58,884,978.61
Net Present Value (NPV):         $16,032,689.66
Internal Rate of Return (IRR):    10.00%
Revenue Gap:                     $28,884,978.61
Discounted Payback Period:        9 years


In [12]:
# Run Scenario C financial analysis using target IRR logic
scenario_c_current, scenario_c_target, rev_c  = enhanced_scenario_analysis(
    name="Scenario C: Rural with Battery",
    discount_rate=0.08,
    initial_capex=195_450_000,
    period_years=10,
    year_1_revenue=30_000_000,     # Original assumption
    year_1_opex=26_697_165,
    revenue_inflation_rate=0.0,
    opex_inflation_rate=0.03,
    target_irr=0.10                # Target IRR to test
)



=== Financial Analysis for: Scenario C: Rural with Battery ===
--- Based on Original Revenue Assumption ($30,000,000.00) ---
Net Present Value (NPV):         $-195,714,722.96
Internal Rate of Return (IRR):    nan%
Total Cost of Ownership (TCO):   $501,503,077.52
Total Net Operating Income:      $-6,053,077.52
Discounted Payback Period:        Does not pay back

--- Revenue Needed to Achieve 10% IRR ---
Required Revenue:                $61,717,357.10
Net Present Value (NPV):         $17,111,324.98
Internal Rate of Return (IRR):    10.00%
Revenue Gap:                     $31,717,357.10
Discounted Payback Period:        9 years
